In [0]:
# Updated copy: table and column metadata is applied by the local ./_common notebook.
%run ./_common

In [0]:
%run ./_gates

In [0]:
# === CONFIG ===
SITE_ID = '<SET_ME>'        # <-- set to your site code, e.g. 'RXX'
if SITE_ID == '<SET_ME>':
    raise ValueError('set SITE_ID')
TARGET  = '5_projects.preborn'

# MSDS common-core sources (12 keys) — point these at YOUR site's MSDS tables.
# (4_prod.raw.* shown as the example Barts layout; a new site edits the right-hand values.)
SRC = {
  "msd001_motherdemo":       "4_prod.raw.msd001motherdemo",
  "msd101_pregbook":         "4_prod.raw.msd101pregbook",
  "msd106_diagnosispreg":    "4_prod.raw.msd106diagnosispreg",
  "msd107_medhistory":       "4_prod.raw.msd107medhistory",
  "msd109_findobsmother":    "4_prod.raw.msd109findobsmother",
  "msd201_carecontactpreg":  "4_prod.raw.msd201carecontactpreg",
  "msd202_careactivitypreg": "4_prod.raw.msd202careactivitypreg",
  "msd301_labdel":           "4_prod.raw.msd301labdel",
  "msd302_careactlabdel":    "4_prod.raw.msd302careactlabdel",
  "msd401_babydemo":         "4_prod.raw.msd401babydemo",
  "msd404_diagneo":          "4_prod.raw.msd404diagneo",
  "msd405_careactbaby":      "4_prod.raw.msd405careactbaby",
}

RACE_CASE_OVERRIDE = None     # None -> reuse the default RACE_CASE crosswalk; or paste a site-specific CASE expr
INCLUDE_ENRICHMENT = False    # enrichment is Barts-only; sites run LIMITED (core+L3)
import re
import uuid
_site_tag = re.sub(r'[^a-z0-9]+', '_', SITE_ID.lower()).strip('_') or 'site'
RUNTAG             = f"{_site_tag}_{uuid.uuid4().hex[:12]}"
RELEASE_DATE       = '2026-07-08'

In [0]:
# === BUILD (core+L3 only into run-unique publish-staging `pub`) ===
assert_mint_parity(SITE_ID)
pub = build_cdm(TARGET, SITE_ID, SRC, include_enrichment=INCLUDE_ENRICHMENT,
                race_case=RACE_CASE_OVERRIDE, runtag=RUNTAG)
print("staged into", pub)

In [0]:
# === VALIDATE + PUBLISH ===
MODE = 'site'

def _validation_results(schema, reconcile_stg=None):
    out = (
        gates_ids(schema, SITE_ID)
        + gates_cdm_structural(schema, SITE_ID)
        + gates_grain(schema, SITE_ID)
        + gates_vocab_fk(schema)
    )
    if reconcile_stg is not None:
        out += gates_reconcile(schema, SITE_ID, reconcile_stg, INCLUDE_ENRICHMENT)
    return out

def _assert_validation(label, schema, reconcile_stg=None):
    checked = _validation_results(schema, reconcile_stg)
    for name, ok, metric in checked:
        print(label, ("PASS" if ok else "FAIL"), name, "|", metric)
    fails = sum(1 for _, ok, _ in checked if not ok)
    print(label, "MODE", MODE, "| fails", fails)
    if fails:
        raise RuntimeError(f"{label}: {fails} gate(s) FAILED")
    return checked

_assert_validation("PRE-PUBLISH", pub, pub + '_stg')

def _post_publish_validate():
    _assert_validation("POST-PUBLISH", TARGET, pub + '_stg')

publish_site(TARGET, pub, SITE_ID, post_validate=_post_publish_validate)
drop_working(TARGET, RUNTAG)
print("PREBORN", SITE_ID, "PUBLISHED")